# Batch correction, Clustering, Phenotyping

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import warnings

from inmoose.pycombat import pycombat_norm, pycombat_seq

In [ ]:
def make_umap(X):
    from umap import UMAP
    embedding = (
        UMAP(random_state=5, n_neighbors=20, min_dist=0.0005, repulsion_strength=1.5)
        .fit_transform(X)
    )
    return embedding

def cluster_cells(X, nclus=40, random_state=0):

    from sklearn.cluster import MiniBatchKMeans, DBSCAN
    from sklearn.feature_extraction.text import TfidfVectorizer
    
    #MiniBatch K-Means for large datasets
    kmeans = MiniBatchKMeans(n_clusters=nclus, batch_size=1000, max_iter=500, random_state=random_state)
    
    kmeans.fit(X)
    
    #get labels
    labels = kmeans.labels_
    labels_str = [str(i) for i in labels]
    
    distances = np.linalg.norm(X - kmeans.cluster_centers_[labels], axis=1)

    return labels_str
    
def knn_psuedobulk(X, nclus=100, random_state=0):
    import numpy as np
    from sklearn.cluster import MiniBatchKMeans
    
    # MiniBatch K-Means for large datasets
    kmeans = MiniBatchKMeans(n_clusters=nclus, batch_size=1000, max_iter=500, random_state=random_state)
    kmeans.fit(X)

    labels = kmeans.labels_
    labels_str = [str(i) for i in labels]
    
    cluster_centers = kmeans.cluster_centers_
    
    return cluster_centers, labels_str

def marker_capping(x, perc = 99):
    x_perc = np.percentile(x, perc)
    return np.where(x > x_perc, x_perc, x) 

## Import

In [ ]:
adata = sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/in/full_data-no_uns.h5ad') #`.uns['spatial']` removed
adata

In [ ]:
# Prep for clustering, essentials only

adata.X = adata.layers['compcounts'].copy()
for i in ['compcounts', 'compexprs', 'exprs']:
    del adata.layers[i]

In [ ]:
adata.obs.major_cell_type.value_counts()

## Normalisation and correction choices

In [ ]:
# Remove adipo, null expression confuses
adata = adata[adata.obs.major_cell_type!='Adipocyte',]

In [ ]:
# Total signal, log1p
adata.obs['log1p_totalsignal'] = np.log1p(adata.X.sum(axis=1))

In [ ]:
# 0-aware norm

#'log1p(X+1/log1p_totalsignal)'
adata.layers['Xpc_totsig_log1p_combat'] = pycombat_norm(
    np.log1p(
        pd.DataFrame(
            pd.DataFrame(adata.X).fillna(0).values.T+1)\
                .div(adata.obs['log1p_totalsignal'].tolist(), axis=1)
    ),
    adata.obs.cohort).T

In [ ]:
adata

## Clustering: all cells

In [ ]:
adata.n_obs

In [ ]:
# new filter: minimum signal
#sns.histplot(adata.obs['log1p_totalsignal'])
log1p_totalsignal__thr = 2.5 #confirmed on umap
adata = adata[adata.obs['log1p_totalsignal']>log1p_totalsignal__thr]

In [ ]:
adata.n_obs

In [ ]:
1384630-1339729
44901/1384630

In [ ]:
# Expression prep

marker_fun = ['aSMA','CD14','CD16','CD163','CD11b','CD31','CD45','CD4','CD68','CD20','CD8','CD56','CD138','Granzyme_B','CD3','CD45RO']

adata.layers['Xpc_totsig_log1p_combat_scaled'] = sc.pp.scale(adata.layers['Xpc_totsig_log1p_combat'])
adata.obsm['Xpc_totsig_log1p_combat_scaled'] = adata[:,marker_fun].layers['Xpc_totsig_log1p_combat_scaled']

In [ ]:
## KNN pseudobulk approach

In [ ]:
exp, cell2clus = knn_psuedobulk(adata.obsm['Xpc_totsig_log1p_combat_scaled'], nclus=5000)

In [ ]:
pd.Series(cell2clus).to_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-allcells.csv')

In [ ]:
adata.obs['cell2clus'] = cell2clus

In [ ]:
import anndata as ad

adata_pb = ad.AnnData(X=exp, obs=pd.DataFrame(index=list(range(len(np.unique(cell2clus))))))
adata_pb.var =  adata[:,marker_fun].var.copy()
adata_pb

In [ ]:
sc.pp.neighbors(adata_pb,random_state=12345)
sc.tl.leiden(adata_pb)
sc.tl.leiden(adata_pb, resolution=2, key_added='leiden_2')
adata_pb.obsm['X_umap'] = make_umap(adata_pb.X)

In [ ]:
cell2clus_cohort = adata.obs[['cell2clus','cohort']].value_counts().reset_index()\
.pivot(index='cell2clus',columns='cohort',values='count').fillna(0).rename_axis(None)
cell2clus_cohort['prop_og'] = cell2clus_cohort['og']/(cell2clus_cohort['og']+cell2clus_cohort['validation'])

adata_pb.obs = adata_pb.obs.join(cell2clus_cohort[['prop_og']])

In [ ]:
adata_pb.obs = adata_pb.obs.join(pd.Series(cell2clus).value_counts())
adata_pb.obs['log10_count'] = np.log10(adata_pb.obs['count'])

In [ ]:
adata_pb.obs = adata_pb.obs.join(adata.obs[['cell2clus','log1p_totalsignal']].groupby('cell2clus').mean())

In [ ]:
with rc_context({"figure.figsize": (7,5),"axes.titlesize": 10}):
    sc.pl.umap(adata_pb, color=['leiden_2','log10_count','prop_og','log1p_totalsignal'], legend_loc='on data', layer='X_scaled', 
        cmap='viridis', legend_fontsize=10, legend_fontoutline=2, frameon=False, s=20, ncols=2)

In [ ]:
# vln signal/batch

clus_='leiden_2'
with rc_context({"figure.figsize": (12, 2.5)}):
    sc.pl.violin(adata_pb,['log1p_totalsignal'],groupby=clus_,stripplot=False,inner="box")
    sc.pl.violin(adata_pb,['prop_og'],groupby=clus_,stripplot=False,inner="box")

In [ ]:
# Pheno umap

with rc_context({"figure.figsize": (5,4),"axes.titlesize": 12}):
    sc.pl.umap(adata_pb, color=marker_fun, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, s=30, ncols=3)

In [ ]:
# Pheno hm
clus_='leiden_2'
sc.pl.matrixplot(adata_pb, marker_fun, clus_, dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
# Import labels

In [ ]:
labels = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/pheno-allcells.csv')
labels.leiden_2=labels.leiden_2.astype(str)

In [ ]:
adata_pb.obs = adata_pb.obs.drop(['lineage','pheno','notes'],axis=1)

In [ ]:
adata_pb.obs = adata_pb.obs.merge(labels)

In [ ]:
# Pheno hm, no QC
clus_='leiden_2'
marker_fun2 = ['aSMA','CD163','CD11b','CD31','CD45','CD4','CD20','CD8','CD56','CD138','CD3']
sc.pl.matrixplot(adata_pb[adata_pb.obs.lineage!='B-T'], 
                 marker_fun2, 'lineage', dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var')

In [ ]:
adata_pb.write('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-allcells.h5ad',compression='gzip')

## Import

In [ ]:
adata.obs['cell2clus'] = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-allcells.csv',index_col=0).astype(str).iloc[:,0].tolist()

In [ ]:
adata_pb = sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-allcells.h5ad')

In [ ]:
labels = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/pheno-allcells.csv')
labels.leiden_2=labels.leiden_2.astype(str)

In [ ]:
adata_pb.obs = adata_pb.obs.drop(['lineage','pheno','notes'],axis=1)

In [ ]:
adata_pb.obs = adata_pb.obs.merge(labels)

## Clustering: T cells

### Pre-subclustering, CD3+, CD4+, CD8+

In [ ]:
adata_t = adata[adata.obs.cell2clus.isin(adata_pb[adata_pb.obs.lineage=='T'].obs_names)]
adata_t.layers['log1p_X'] = np.log1p(adata_t.X)
adata_t.layers['Xpc_totsig_log1p_combat_scaled'] = sc.pp.scale(adata_t.layers['Xpc_totsig_log1p_combat'])
adata_t.n_obs/adata.n_obs*100

In [ ]:
adata_t.obs['clus_tsub'] = cluster_cells(sc.pp.scale(adata_t[:,['CD3','CD4','CD8','FoxP3']].layers['Xpc_totsig_log1p_combat']), nclus=5)

In [ ]:
# vln markers/batch

with rc_context({"figure.figsize": (3,2)}):
    sc.pl.violin(adata_t, ['CD3','CD4','CD8','FoxP3','log1p_totalsignal'],
                 groupby='clus_tsub',stripplot=False,layer='Xpc_totsig_log1p_combat',inner="box")

In [ ]:
x='log1p_totalsignal'
#x='CD3'
#y='FoxP3'
layer='Xpc_totsig_log1p_combat'
for y in ['CD3','CD4','CD8','FoxP3']:
    fig,ax=plt.subplots(figsize=(3,3))
    sc.pl.scatter(adata_t, x=x, y=y, color='clus_tsub', alpha=0.25, layers=layer, ax=ax)

In [ ]:
# remove CD3- cluster (non-T cells)

In [ ]:
adata_t = adata_t[adata_t.obs.clus_tsub!='1']

In [ ]:
# CD4/CD8 cluster

In [ ]:
df_ = sc.get.obs_df(adata_t, ['CD4','CD8','CD3'], layer='Xpc_totsig_log1p_combat')

In [ ]:
fig,ax=plt.subplots(figsize=(7,7))
#plt.xticks([i/10 for i in range(-20,80,3)])
#plt.yticks([i/10 for i in range(-20,70,3)])
#plt.grid(True)
sns.scatterplot(df_, x='CD4', y='CD8', alpha=0.1, s=1, color='red', ax=ax)
ax.tick_params(axis='x', rotation=90)

x1,y1 = -0.8,-0.9
x2,y2 = 1,-0.5

slope = (y2 - y1) / (x2 - x1)
intercept = y1 - slope * x1
x_extended = np.linspace(-2, 8, 100)
y_extended = slope * x_extended + intercept
plt.plot(x_extended, y_extended, 'b-', linewidth=0.5, label='Extended line')

plt.show()

In [ ]:
df_['subset'] = ''
for i_,r_ in df_.iterrows():
    res = (x2 - x1) * (r_['CD8'] - y1) - (y2 - y1) * (r_['CD4'] - x1)
    if res > 0:
        df_.loc[i_,'subset']='CD8'
    else:
        df_.loc[i_,'subset']='CD4'

In [ ]:
fig,ax=plt.subplots(figsize=(3,3))
sns.scatterplot(df_, x='CD4', y='CD8', hue='subset', alpha=0.1, s=1, ax=ax)
plt.show()

In [ ]:
adata_t.obs['subset'] = df_['subset'].tolist()

In [ ]:
adata_t.write('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_t.h5ad',compression='gzip')

### Re-psuedobulking, re-clustering - CD8+

Aiming for:
* CD8 Tmem
* CD8 GZMB Tmem
* CD8 Tnaive
* CD8 Tex

In [ ]:
adata_CD8 = adata_t[adata_t.obs.subset=='CD8']
adata_CD8.n_obs/adata_t.n_obs*100

In [ ]:
adata.var_names

In [ ]:
markers_CD8 = ['CD45RO','CD127','Granzyme_B','PD-1','TIM-3'] #'TIGIT','CCR7' #batch...

In [ ]:
# Expression prep - scaling
adata_CD8.layers['Xpc_totsig_log1p_combat_scaled'] = sc.pp.scale(adata_CD8.layers['Xpc_totsig_log1p_combat'])
adata_CD8.obsm['Xpc_totsig_log1p_combat_scaled'] = adata_CD8[:,markers_].layers['Xpc_totsig_log1p_combat_scaled']

In [ ]:
### KNN pseudobulk approach

In [ ]:
adata_in = adata_CD8

In [ ]:
exp, cell2clus = knn_psuedobulk(adata_in.obsm['Xpc_totsig_log1p_combat_scaled'], nclus=1000)

In [ ]:
pd.Series(cell2clus).to_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-CD8.csv')

In [ ]:
adata_in.obs['cell2clus'] = cell2clus

In [ ]:
import anndata as ad

adata_t_pb = ad.AnnData(X=exp, obs=pd.DataFrame(index=list(range(len(np.unique(cell2clus))))))
adata_t_pb.var =  adata_in[:,markers_].var.copy()
adata_t_pb

In [ ]:
sc.pp.neighbors(adata_t_pb,random_state=12345)
sc.tl.leiden(adata_t_pb)
sc.tl.leiden(adata_t_pb, resolution=0.5, key_added='leiden_0.5')
adata_t_pb.obsm['X_umap'] = make_umap(adata_t_pb.X)

In [ ]:
cell2clus_cohort = adata_in.obs[['cell2clus','cohort']].value_counts().reset_index()\
.pivot(index='cell2clus',columns='cohort',values='count').fillna(0).rename_axis(None)
cell2clus_cohort['prop_og'] = cell2clus_cohort['og']/(cell2clus_cohort['og']+cell2clus_cohort['validation'])

adata_t_pb.obs = adata_t_pb.obs.join(cell2clus_cohort[['prop_og']])

adata_t_pb.obs = adata_t_pb.obs.join(pd.Series(cell2clus).value_counts())
adata_t_pb.obs['log10_count'] = np.log10(adata_t_pb.obs['count'])

adata_t_pb.obs = adata_t_pb.obs.join(adata.obs[['cell2clus','log1p_totalsignal']].groupby('cell2clus').mean())

In [ ]:
with rc_context({"figure.figsize": (4,3),"axes.titlesize": 10}):
    sc.pl.umap(adata_CD8_pb, color=['leiden','log1p_totalsignal','prop_og'], legend_loc='on data', layer='X_scaled',  ##'log10_count','log1p_totalsignal'
        cmap='viridis', legend_fontsize=12, legend_fontoutline=2, frameon=False, s=50, ncols=3)

In [ ]:
# vln signal/batch
clus_='leiden'
with rc_context({"figure.figsize": (8, 2.5)}):
    for i in ['log1p_totalsignal','prop_og']: ##,'CD3','CD8','CD4','FoxP3']: 
        sc.pl.violin(adata_t_pb,i,groupby=clus_,stripplot=False,inner="box")

In [ ]:
# Pheno umap

with rc_context({"figure.figsize": (4,3),"axes.titlesize": 18}):
    sc.pl.umap(adata_t_pb, color=markers_CD8, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=3)

In [ ]:
# Pheno hm
clus_='leiden'
sc.pl.matrixplot(adata_t_pb, markers_CD8, clus_, dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
# putative naming
renames_=['CD8_Other','CD8_GZMB','CD8_Tex','CD8_GZMB','CD8_naive','CD8_naive','CD8_Tmem','CD8_Tmem','CD8_naive','CD8_Tex','CD8_Tex']
renames_=dict(zip(list(range(adata_t_pb.obs.leiden.nunique())),renames_))

In [ ]:
adata_CD8_pb.obs['ph'] = [renames_[int(i)] for i in adata_CD8_pb.obs['leiden']]

In [ ]:
# Pheno hm
clus_='ph'
sc.pl.matrixplot(adata_CD8_pb, markers_CD8, clus_, dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var')

In [ ]:
adata_in.obs = adata_in.obs.merge(
    adata_t_pb.obs[['ph']].reset_index().rename({'index':'cell2clus'},axis=1)
)

In [ ]:
adata_CD8_pb = adata_t_pb.copy()

In [ ]:
adata_CD8_pb.write('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-CD8.h5ad',compression='gzip')

### Re-psuedobulking, re-clustering - CD4+

Aiming for:
* CD4 Tex
* CD4 Tmem
* CD4 Treg
* CD4 naive
* CD4 Th17

In [ ]:
adata_CD4 = adata_t[adata_t.obs.subset=='CD4']
adata_CD4.n_obs/adata_t.n_obs*100

In [ ]:
adata.var_names

In [ ]:
markers_CD4 = ['CD45RO','CD127','Granzyme_B','PD-1','FoxP3','CCR6'] #'TIGIT','CCR7' #batch...

In [ ]:
# Expression prep - scaling
adata_CD4.layers['Xpc_totsig_log1p_combat_scaled'] = sc.pp.scale(adata_CD4.layers['Xpc_totsig_log1p_combat'])
adata_CD4.obsm['Xpc_totsig_log1p_combat_scaled'] = adata_CD4[:,markers_].layers['Xpc_totsig_log1p_combat_scaled']

In [ ]:
### KNN pseudobulk approach

In [ ]:
adata_in = adata_CD4

In [ ]:
exp, cell2clus = knn_psuedobulk(adata_in.obsm['Xpc_totsig_log1p_combat_scaled'], nclus=1000)

In [ ]:
pd.Series(cell2clus).to_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-CD4.csv')

In [ ]:
adata_in.obs['cell2clus'] = cell2clus

In [ ]:
import anndata as ad

adata_t_pb = ad.AnnData(X=exp, obs=pd.DataFrame(index=list(range(len(np.unique(cell2clus))))))
adata_t_pb.var =  adata_in[:,markers_].var.copy()
adata_t_pb

In [ ]:
sc.pp.neighbors(adata_t_pb,random_state=12345)
sc.tl.leiden(adata_t_pb)
sc.tl.leiden(adata_t_pb, resolution=0.5, key_added='leiden_0.5')
adata_t_pb.obsm['X_umap'] = make_umap(adata_t_pb.X)

In [ ]:
cell2clus_cohort = adata_in.obs[['cell2clus','cohort']].value_counts().reset_index()\
.pivot(index='cell2clus',columns='cohort',values='count').fillna(0).rename_axis(None)
cell2clus_cohort['prop_og'] = cell2clus_cohort['og']/(cell2clus_cohort['og']+cell2clus_cohort['validation'])

adata_t_pb.obs = adata_t_pb.obs.join(cell2clus_cohort[['prop_og']])

adata_t_pb.obs = adata_t_pb.obs.join(pd.Series(cell2clus).value_counts())
adata_t_pb.obs['log10_count'] = np.log10(adata_t_pb.obs['count'])

adata_t_pb.obs = adata_t_pb.obs.join(adata.obs[['cell2clus','log1p_totalsignal']].groupby('cell2clus').mean())

In [ ]:
with rc_context({"figure.figsize": (4,3),"axes.titlesize": 10}):
    sc.pl.umap(adata_CD4_pb, color=['leiden','log1p_totalsignal','prop_og'], legend_loc='on data', layer='X_scaled',  ##'log10_count','log1p_totalsignal'
        cmap='viridis', legend_fontsize=12, legend_fontoutline=2, frameon=False, s=50, ncols=3)

In [ ]:
# vln signal/batch
clus_='leiden'
with rc_context({"figure.figsize": (8, 2.5)}):
    for i in ['log1p_totalsignal','prop_og']: ##,'CD3','CD8','CD4','FoxP3']: 
        sc.pl.violin(adata_t_pb,i,groupby=clus_,stripplot=False,inner="box")

In [ ]:
# Pheno umap

with rc_context({"figure.figsize": (4,3),"axes.titlesize": 15}):
    sc.pl.umap(adata_t_pb, color=markers_CD4, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=3)

In [ ]:
# Pheno hm
clus_='leiden'
sc.pl.matrixplot(adata_t_pb, markers_CD4, clus_, dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

Aiming for:
* CD4 Tex
* CD4 Tmem
* CD4 Treg
* CD4 naive
* CD4 Th17

In [ ]:
# putative naming
renames_=['CD4_Other','CD4_CCR6','CD4_Treg','CD4_GZMB','CD4_GZMB',#4
          'CD4_Treg','CD4_Treg','CD4_Tn','CD4_Tn','CD4_CCR6',#9
          'CD4_Tmem','CD4_Treg','CD4_GZMB','CD4_Tex']
renames_=dict(zip(list(range(adata_t_pb.obs.leiden.nunique())),renames_))

In [ ]:
adata_t_pb.obs['ph'] = [renames_[int(i)] for i in adata_t_pb.obs['leiden']]

In [ ]:
# Pheno hm
clus_='ph'
sc.pl.matrixplot(adata_t_pb, markers_CD4, clus_, dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var')

In [ ]:
adata_CD4_pb = adata_t_pb.copy()

In [ ]:
adata_CD4_pb.write('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-CD4.h5ad',compression='gzip')

## Clustering: mixed myeloid

In [ ]:
# Myeloid
#contains mix of CD45only; true myeloid, Mac, other unknowns

Aiming for:
* M1/M2 Mac
* "Myeloid/MDSC"
* MoDC
* Mono

In [ ]:
adata_mye = adata[adata.obs.cell2clus.isin(adata_pb[adata_pb.obs.lineage=='Myeloid'].obs_names)]
adata_mye.layers['Xpc_totsig_log1p_combat_scaled'] = sc.pp.scale(adata_mye.layers['Xpc_totsig_log1p_combat'])
adata_mye.n_obs/adata.n_obs*100

In [ ]:
adata.var_names

In [ ]:
markers_mye = ['CD14','CD16','CD163','CD11b','CD11c','CD68','HLA-DR',] #PD-L1=batch

In [ ]:
for m in markers_mye:
    fig,ax=plt.subplots(figsize=(3,3))
    sc.pl.scatter(adata_mye, x='CD11b', y=m, color='cohort', alpha=0.25, layers=layer, ax=ax)

In [ ]:
# Expression prep - scaling
adata_mye.obsm['Xpc_totsig_log1p_combat_scaled'] = adata_mye[:,markers_mye].layers['Xpc_totsig_log1p_combat_scaled']

In [ ]:
### KNN pseudobulk approach

In [ ]:
adata_in = adata_mye

In [ ]:
exp, cell2clus = knn_psuedobulk(adata_in.obsm['Xpc_totsig_log1p_combat_scaled'], nclus=1000)

In [ ]:
pd.Series(cell2clus).to_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-mye.csv')

In [ ]:
adata_in.obs['cell2clus'] = cell2clus

In [ ]:
import anndata as ad

adata_m_pb = ad.AnnData(X=exp, obs=pd.DataFrame(index=list(range(len(np.unique(cell2clus))))))
adata_m_pb.var =  adata_in[:,markers_mye].var.copy()
adata_m_pb

In [ ]:
sc.pp.neighbors(adata_m_pb,random_state=12345)
sc.tl.leiden(adata_m_pb)
sc.tl.leiden(adata_m_pb, resolution=0.5, key_added='leiden_0.5')
sc.tl.leiden(adata_m_pb, resolution=1.5, key_added='leiden_1.5')
adata_m_pb.obsm['X_umap'] = make_umap(adata_m_pb.X)

In [ ]:
cell2clus_cohort = adata_in.obs[['cell2clus','cohort']].value_counts().reset_index()\
.pivot(index='cell2clus',columns='cohort',values='count').fillna(0).rename_axis(None)
cell2clus_cohort['prop_og'] = cell2clus_cohort['og']/(cell2clus_cohort['og']+cell2clus_cohort['validation'])

adata_m_pb.obs = adata_m_pb.obs.join(cell2clus_cohort[['prop_og']])

adata_m_pb.obs = adata_m_pb.obs.join(pd.Series(cell2clus).value_counts())
adata_m_pb.obs['log10_count'] = np.log10(adata_m_pb.obs['count'])

adata_m_pb.obs = adata_m_pb.obs.join(adata.obs[['cell2clus','log1p_totalsignal']].groupby('cell2clus').mean())

In [ ]:
with rc_context({"figure.figsize": (4,4),"axes.titlesize": 10}):
    sc.pl.umap(adata_m_pb, color=['leiden_1.5','log1p_totalsignal','prop_og'], legend_loc='on data', layer='X_scaled',  ##'log10_count','log1p_totalsignal'
        cmap='viridis', legend_fontsize=12, legend_fontoutline=2, frameon=False, s=50, ncols=3)

In [ ]:
# vln signal/batch
clus_='leiden_1.5'
with rc_context({"figure.figsize": (8, 2.5)}):
    for i in ['log1p_totalsignal','prop_og']: ##,'CD3','CD8','CD4','FoxP3']: 
        sc.pl.violin(adata_m_pb,i,groupby=clus_,stripplot=False,inner="box")

In [ ]:
# Pheno umap

with rc_context({"figure.figsize": (3,3),"axes.titlesize": 18}):
    sc.pl.umap(adata_m_pb, color=markers_mye, legend_loc='right', legend_fontsize=10, legend_fontoutline=2, 
               cmap='viridis', frameon=False, ncols=4)

In [ ]:
# Pheno hm
clus_='leiden_1.5'
sc.pl.matrixplot(adata_m_pb, markers_mye, clus_, dendrogram=True, cmap="Reds",swap_axes=True,standard_scale='var')

In [ ]:
# putative naming
renames_=['Mac_M1','Mye_other','Mye_other','Mye_HLADR',
          'Mye_other','DC','Mac_M2','Mono_CD14',
          'Mye_other','Mac_M2','Mono_CD16','Mono_CD16',
          'Mac_M1','Mono_CD16','Mye_other']

renames_=dict(zip(list(range(adata_m_pb.obs['leiden_1.5'].nunique())),renames_))

In [ ]:
adata_m_pb.obs['ph'] = [renames_[int(i)] for i in adata_m_pb.obs['leiden_1.5']]

In [ ]:
# Pheno hm
clus_='ph'
sc.pl.matrixplot(adata_m_pb, markers_mye, clus_, dendrogram=False, cmap="Reds",swap_axes=False,standard_scale='var')

In [ ]:
adata_m_pb.write('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-mye.h5ad',compression='gzip')

## Output

* major_cell_type
* minor_cell_type
* functional

In [ ]:
adata_ = adata.copy()

#adata.write('')

In [ ]:
adata_.obs.shape[0]

In [ ]:
obs_og_all =  sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/in/full_data-no_uns.h5ad').obs

In [ ]:
obs_og = sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/in/full_data-no_uns.h5ad').obs
obs_og = obs_og[['sample_id','Cell_ID']]

In [ ]:
obs_og.shape[0]

In [ ]:
adata_pb = sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-allcells.h5ad')
labels = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/pheno-allcells.csv')
labels.leiden_2=labels.leiden_2.astype(str)
adata_pb.obs = adata_pb.obs.drop(['lineage','pheno','notes'],axis=1)
adata_pb.obs = adata_pb.obs.merge(labels)

In [ ]:
adata.obs['cell2clus'] = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-allcells.csv',index_col=0).astype(str).iloc[:,0].tolist()
obs_all = adata.obs.merge(
    adata_pb.obs[['lineage','pheno']].reset_index().rename({'index':'cell2clus'},axis=1)
)[['sample_id','Cell_ID','lineage','pheno']]

In [ ]:
obs_t = sc.read("/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_t.h5ad").obs

In [ ]:
obs_ = obs_t[obs_t.subset=='CD8']
obs_['cell2clus'] = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-CD8.csv',index_col=0).iloc[:,0].astype(str).tolist()
obs_ = obs_.merge(
    sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-CD8.h5ad').obs[['ph']].reset_index().rename({'index':'cell2clus'},axis=1)
)
obs_cd8 = obs_[['sample_id','Cell_ID','ph']].copy()

In [ ]:
obs_ = obs_t[obs_t.subset=='CD4']
obs_['cell2clus'] = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-CD4.csv',index_col=0).iloc[:,0].astype(str).tolist()
obs_ = obs_.merge(
    sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-CD4.h5ad').obs[['ph']].reset_index().rename({'index':'cell2clus'},axis=1)
)
obs_cd4 = obs_[['sample_id','Cell_ID','ph']].copy()

In [ ]:
adata.obs['lineage']=obs_all['lineage'].tolist()
obs_m = adata[adata.obs.lineage=='Myeloid'].obs

In [ ]:
obs_ = obs_m.copy()
obs_['cell2clus'] = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/cell2clus-mye.csv',index_col=0).iloc[:,0].astype(str).tolist()
obs_ = obs_.merge(
    sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/adata_pb-mye.h5ad').obs[['ph']].reset_index().rename({'index':'cell2clus'},axis=1)
)
obs_mye = obs_[['sample_id','Cell_ID','ph']].copy()

In [ ]:
obs_out = obs_og.merge(obs_all[['sample_id','Cell_ID','lineage']], how='outer')

In [ ]:
obs_out.lineage = obs_out.lineage.fillna('Adipocyte')

In [ ]:
obs_out = obs_out.merge(pd.concat([obs_cd4,obs_cd8,obs_mye],axis=0), how='outer')

In [ ]:
# issue: the log1p_totalsignal cells
# rename to Lin-

adata_ = sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/in/full_data-no_uns.h5ad') #`.uns['spatial']` removed
adata_ = adata_[adata_.obs.major_cell_type!='Adipocyte',]
adata_.obs['log1p_totalsignal'] = np.log1p(adata_.X.sum(axis=1))
log1p_totalsignal__thr = 2.5 #confirmed on umap
min_signal_obs = adata_[adata_.obs['log1p_totalsignal']<=2.5].obs[['sample_id','Cell_ID']]
##del adata_

min_signal_obs['min_signal'] = True

obs_out = obs_out.merge(min_signal_obs,how='outer')

obs_out['lineage'] = np.where(obs_out['min_signal'].fillna(False),'Lin-',obs_out['lineage'])
obs_out['ph'] = np.where(obs_out['min_signal'].fillna(False),'Lin-',obs_out['ph'])
obs_out = obs_out.drop('min_signal',axis=1)

In [ ]:
# CD138+ to PC

df_PC_Bcell_gating = pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/df_PC_Bcell_gating.csv')[['sample_id','Cell_ID','lineage2']]
obs_out = obs_out.merge(df_PC_Bcell_gating, how='outer')

obs_out['lineage'] = obs_out.lineage2.fillna(obs_out.lineage)
obs_out = obs_out.drop('lineage2',axis=1)

In [ ]:
# Ki-67+
exp_ = adata[:,'Ki-67'].layers['Xpc_totsig_log1p_combat'].ravel()
adata.obs['Ki67_pos'] = (exp_ > np.mean(exp_)+np.std(exp_)).astype(str)

obs_out = obs_out.merge(adata.obs[['sample_id','Cell_ID','Ki67_pos']], how='outer')
obs_out.Ki67_pos = obs_out.Ki67_pos.fillna(False)

In [ ]:
# Fill in ph with lineage

obs_out.ph = obs_out.ph.fillna(obs_out.lineage)

In [ ]:
# Manuel fixes

# Remove lineage==T, Tsubclus==Artifact (CD3-CD8-CD4- FoxP3+)
obs_out.lineage = np.where(obs_out.ph=='T','Artifact',obs_out.lineage)
obs_out.ph = np.where(obs_out.ph=='T','Artifact',obs_out.ph)

# Mis-labeled PC/Lin-
obs_out.lineage = np.where( (obs_out.lineage=='Myeloma') & (obs_out.ph=='Lin-'), 'Lin-', obs_out.lineage )

In [ ]:
#sns.heatmap(pd.crosstab(obs_out.ph,obs_out.lineage)>0)

In [ ]:
obs_out.to_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/v2_pheno-labels_update3.csv')